# Stage A — Feature Expansion Diagnostic Inspection

This notebook validates the expanded feature schema (v0.2.0 → v0.3.0) after the
Phase 1–5 feature engineering improvements:

1. **Block restructuring**: `cohort_specific` → 4 domain-specific blocks (insight, autism_profile, hostility_aggression, treatment_resistance)
2. **Subscale wiring**: BDHI 9, BRIEF 9, RBS-R 6, ADI-R 4, LSAS 2, WAIS-IV ICV, SUMD 4 additional items
3. **PANSS Wallwork 5-factor model** (SZ)
4. **Derived composites**: polypharmacy index, illness burden, onset category, waist/height ratio

**Diagnostic goals:**
- Per-cohort feature coverage heatmap
- Feature distribution violin plots per new block
- Inter-feature correlation heatmaps (redundancy check)
- Trans-cohort feature selection before/after
- Missingness pattern analysis
- t-SNE embedding colored by new features

## 0. Setup

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

repo = Path.cwd().resolve()
if repo.name == 'notebooks':
    repo = repo.parent
sys.path.insert(0, str(repo / 'src'))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Project dark theme
DARK = dict(template='plotly_dark', paper_bgcolor='#0f1117', plot_bgcolor='#1a1d27')
SCALE = 3
COHORT_COLORS = {'bp': '#6366f1', 'sz': '#f87171', 'asp': '#34d399'}
COHORT_NAMES = {'bp': 'Bipolar', 'sz': 'Schizophrenia', 'asp': 'Autism'}

OUT = repo / 'output' / 'stratification' / 'stage_a_inspection'
OUT.mkdir(parents=True, exist_ok=True)
print(f'Repo: {repo}')
print(f'Output: {OUT}')

Repo: /Users/andriikulakovskyi/Desktop/llm-rl/psych-dataset
Output: /Users/andriikulakovskyi/Desktop/llm-rl/psych-dataset/output/stratification/stage_a_inspection


## 1. Load expanded schema and build harmonized dataset

In [2]:
from face_stratification.harmonization.feature_schema import load_feature_schema
from face_stratification.harmonization.harmonizer import build_harmonized_dataset

schema = load_feature_schema()
print(f'Schema version: {schema.version}')
print(f'Blocks: {len(schema.blocks)}')
print(f'Features: {len(schema.features)}')
print()

for b in schema.blocks:
    n = sum(1 for f in schema.features if f.block == b.id)
    print(f'  {b.id:25s} {n:3d} features  [{b.metric}, min_frac={b.min_fraction_present}]')

Schema version: 0.2.0
Blocks: 21
Features: 184

  demographics                5 features  [gower, min_frac=0.6]
  mood                        9 features  [cosine, min_frac=0.5]
  psychosis                  11 features  [cosine, min_frac=0.6]
  anxiety_impulsivity         8 features  [cosine, min_frac=0.2]
  functioning                 4 features  [cosine, min_frac=0.25]
  sleep_circadian             3 features  [cosine, min_frac=0.5]
  cognition                  17 features  [euclidean, min_frac=0.15]
  biology                    13 features  [euclidean, min_frac=0.5]
  treatment                  11 features  [cosine, min_frac=0.5]
  substance                   5 features  [gower, min_frac=0.6]
  trauma                      7 features  [cosine, min_frac=0.5]
  family_history              4 features  [gower, min_frac=0.5]
  comorbidities               2 features  [gower, min_frac=0.5]
  suicide_history             4 features  [gower, min_frac=0.5]
  psychiatric_history         8 feature

In [4]:
csv_paths = {c: repo / 'data' / f'{c.upper()}.csv' for c in ('bp', 'sz', 'asp')}
ds = build_harmonized_dataset(csv_paths, schema=schema, max_rows_per_cohort=300)

cohort = ds.metadata['cohort']
print(f'Patients: {ds.n_patients}, Features: {ds.n_features}')
print(f'Global NaN rate: {ds.X.isna().mean().mean():.1%}')
print(f'Cohort distribution:')
for c, n in cohort.value_counts().sort_index().items():
    nan_rate = ds.X.loc[cohort == c].isna().mean().mean()
    print(f'  {c}: {n} patients, NaN rate: {nan_rate:.1%}')

Patients: 900, Features: 184
Global NaN rate: 70.9%
Cohort distribution:
  asp: 300 patients, NaN rate: 87.7%
  bp: 300 patients, NaN rate: 58.5%
  sz: 300 patients, NaN rate: 66.6%


## 2. Per-cohort feature coverage heatmap

Shows % non-NaN for each feature in each cohort. Features grouped by block, cohorts as columns.
Identifies which new subscales have usable coverage and which are sparse.

In [5]:
# Compute per-cohort coverage
coverage = pd.DataFrame(
    {c: ds.X.loc[cohort == c].notna().mean() for c in ('bp', 'sz', 'asp')}
)

# Order features by block
block_order = [b.id for b in schema.blocks]
feat_blocks = {f.id: f.block for f in schema.features}
feat_order = sorted(coverage.index, key=lambda f: (block_order.index(feat_blocks.get(f, block_order[-1])), f))
coverage = coverage.loc[feat_order]

# Add block annotations
block_labels = [feat_blocks.get(f, '?') for f in feat_order]

fig = go.Figure(data=go.Heatmap(
    z=coverage.values,
    x=['BP', 'SZ', 'ASP'],
    y=[f'{feat_blocks.get(f, "?")}:{f}' for f in feat_order],
    colorscale='RdYlGn',
    zmin=0, zmax=1,
    colorbar=dict(title='Coverage', tickformat='.0%'),
    hovertemplate='%{y}<br>%{x}: %{z:.1%}<extra></extra>',
))
fig.update_layout(
    title='Per-cohort feature coverage (% non-NaN)',
    height=max(400, len(feat_order) * 14),
    width=700,
    yaxis=dict(dtick=1, tickfont=dict(size=7)),
    **DARK,
)
fig.write_image(str(OUT / 'fig01_coverage_heatmap.png'), width=700, height=max(400, len(feat_order) * 14), scale=SCALE)
fig.show()

In [6]:
# Summary: features with >50% coverage in each cohort (trans-cohort candidates)
threshold = 0.5
eligible = coverage.min(axis=1)
trans_cohort_candidates = eligible[eligible >= threshold].index.tolist()
print(f'Features with ≥{threshold:.0%} coverage in ALL 3 cohorts (trans-cohort): {len(trans_cohort_candidates)}')
for f in trans_cohort_candidates:
    covs = ', '.join(f'{c}={coverage.loc[f, c]:.0%}' for c in ('bp', 'sz', 'asp'))
    print(f'  {f:45s} {covs}')

Features with ≥50% coverage in ALL 4 cohorts (trans-cohort): 9
  demo_age_years                                bp=100%, sz=100%, dr=100%, asp=100%
  demo_sex_male                                 bp=100%, sz=100%, dr=100%, asp=100%
  tx_polypharmacy_index                         bp=100%, sz=100%, dr=100%, asp=100%
  sub_alcohol_current                           bp=100%, sz=100%, dr=100%, asp=100%
  sub_cannabis_current                          bp=100%, sz=100%, dr=100%, asp=100%
  sub_tobacco_current                           bp=100%, sz=100%, dr=100%, asp=100%
  sub_use_disorder                              bp=100%, sz=100%, dr=100%, asp=100%
  cm_n_psychiatric                              bp=100%, sz=100%, dr=100%, asp=100%
  cm_n_somatic                                  bp=100%, sz=100%, dr=100%, asp=100%


## 3. Block catalog and feature distributions

The 184 features are organized into **21 clinical blocks**. Each block generates its own
similarity graph layer with a domain-appropriate distance metric (cosine, euclidean, or
Gower) and minimum overlap constraints. Below is the full block catalog, followed by
distribution violin plots for every block.

---

### Block descriptions

**demographics** (5 features — BP, SZ, ASP) · metric: gower
Age, sex, education level, marital status, employment status. These are static or slowly
changing social determinants. Gower distance handles the mix of continuous (age),
ordinal (education), and binary (sex, partnered, employed) types.

**mood** (9 features — BP, SZ, ASP) · metric: cosine
Clinician-rated and self-rated mood symptom severity. MADRS and YMRS (BP),
CGI-S (BP, SZ), QIDS and MAThyS (BP), ASRM (BP), BDI-II (ASP),
Calgary depression in schizophrenia (SZ). Different cohorts
use different depression/mania instruments, but all contribute to the same mood
similarity layer.

**psychosis** (11 features — SZ only) · metric: cosine
PANSS total and 3 traditional subscales (positive, negative, general), the Wallwork
5-factor decomposition (positive, negative, disorganized, excited, depressed), plus
AIMS involuntary movements and BARS akathisia. The Wallwork factors capture
clinically distinct psychotic dimensions — notably the depressed factor is
near-orthogonal to positive/negative and the excited factor captures agitation
independently.

**anxiety_impulsivity** (8 features — BP, ASP) · metric: cosine
State anxiety (STAI-YA), clinician-rated anxiety (HAM-A for ASP), social anxiety
(LSAS total for ASP), total impulsivity (BIS-10) and its 3 dimensions (attentional,
motor, non-planning), and affective lability (ALS for BP). Impulsivity structure is
prognostically distinct from total impulsivity — attentional impulsivity predicts
cognitive outcomes while motor impulsivity predicts behavioral risk.

**hostility_aggression** (10 features — BP only) · metric: cosine
Buss-Durkee Hostility Inventory (BDHI) total and 9 subscales: assault, indirect
hostility, irritability, negativism, resentment, suspicion, verbal hostility, guilt,
and attitudinal hostility. Separated from anxiety_impulsivity because hostile
attribution bias is a distinct clinical dimension from impulsivity — patients can be
impulsive without being hostile and vice versa.

**functioning** (4 features — BP, SZ, ASP) · metric: cosine
Functional impairment and quality of life: FAST (BP), PSP (SZ), EQ-5D
(BP, SZ, ASP), EGF (ASP). Each cohort uses different primary functioning
scales, but all measure the same construct — daily-life disability and health utility.

**sleep_circadian** (3 features — BP, SZ, ASP) · metric: cosine
Sleep quality (PSQI, all cohorts), daytime somnolence (ESS), and chronotype
(CSM, BP). Sleep disturbance is a transdiagnostic risk factor that cuts across
all pathologies and often predicts relapse.

**cognition** (17 features — BP, SZ) · metric: euclidean
The core neuropsychological battery: TMT-A/B times and derived indices (B−A cost,
B/A ratio), Stroop components (word, color, color-word, interference), CVLT
learning, short/long delay recall and recognition, phonemic and semantic fluency,
WAIS similarities, vocabulary, and working memory. Euclidean distance is used
because cognitive scores are already on comparable scales after z-scoring and the
profile shape matters more than angular similarity.

**neuropsych** (10 features — BP, SZ) · metric: euclidean
Extended neuropsychological battery beyond the core: WAIS-IV subtests (matrices,
code, symbol search), digit span (forward, backward, total), and CPT sustained
attention (omissions, commissions, hit RT, variability). These are available
only for BP and SZ which administer the full WAIS and CPT.

**biology** (13 features — BP, SZ, ASP) · metric: euclidean
Somatic health panel: BMI, waist circumference, blood pressure (systolic/diastolic),
heart rate, QTc interval, fasting glucose, lipid panel (total cholesterol, HDL,
triglycerides, TG/HDL ratio), plus derived metabolic syndrome flag (IDF/ATP-III
criteria) and waist-to-height ratio. Euclidean distance captures the cardiometabolic
risk profile.

**treatment** (11 features — BP, SZ, ASP) · metric: cosine
Current psychotropic medication classes (binary: antidepressant, antipsychotic,
mood stabilizer, benzodiazepine, lithium, clozapine), medication adherence (MARS),
plasma levels (lithium, valproate, clozapine), and the derived polypharmacy index
(count of concurrent medication classes — present in all 3 cohorts, trans-cohort
eligible).

**substance** (5 features — BP, SZ, ASP) · metric: gower
Current tobacco use and cigarettes/day, current alcohol and cannabis use, and
lifetime substance use disorder. Gower distance handles the mix of binary (yes/no)
and continuous (CPD) types. All 5 features are measured across all 3 cohorts.

**trauma** (7 features — BP, SZ, ASP) · metric: cosine
Childhood Trauma Questionnaire total and 5 subscales (emotional abuse, physical
abuse, sexual abuse, emotional neglect, physical neglect), plus PCL-5 PTSD.
The 5 CTQ subscales have differential associations with specific
disorders and treatment response — sexual abuse has distinct biological signatures
from emotional neglect, for example.

**family_history** (4 features — BP, SZ, ASP) · metric: gower
Binary flags for any family history of bipolar disorder, suicide, or substance use,
plus count of affected relatives across the pedigree. ASP does not extract family
history (all NaN). Gower handles the binary/count mix.

**comorbidities** (2 features — BP, SZ, ASP) · metric: gower
Count of somatic and psychiatric comorbidities. Simple burden measures that capture
multimorbidity load.

**suicide_history** (4 features — BP, SZ, ASP) · metric: gower
Lifetime suicidal ideation (binary), lifetime attempt (binary), number of attempts,
and violent attempt flag. ASP has partial coverage (BDI-II item 9 only for
ideation). This block feeds into the safety analysis that checks no cluster
concentrates high-risk patients.

**psychiatric_history** (8 features — BP, SZ) · metric: cosine
Age at first episode, illness duration in years, lifetime episode counts (depressive
and manic for BP), rapid cycling flag (BP), number of hospitalizations, derived
onset category (early/typical/late), and cumulative illness burden composite
(log-scaled duration × episodes × hospitalizations).

**insight** (10 features — SZ only) · metric: cosine
Scale to Assess Unawareness of Mental Disorder (SUMD): mean insight score plus 9
individual awareness items (illness, medication effect, social consequences,
hallucinations, delusions, thought disorder, flat affect, anhedonia, asociality).
Item-level insight profiles distinguish clinical subtypes — patients who are aware
of hallucinations but unaware of negative symptoms represent a different phenotype
from globally poor insight.

**autism_profile** (32 features — ASP only) · metric: cosine
Comprehensive autism phenotyping: DSM-5 domain criteria met (social communication,
restricted behaviors), developmental milestone (age at first phrases), ADI-R 4
diagnostic domains (social interaction, communication, restricted behaviors,
abnormal development), RBS-R total and 6 repetitive behavior domains (stereotypies,
self-injury, compulsive, rituals, sameness, restricted), BRIEF executive composite
and 9 subscales (inhibition, flexibility, emotional control, self-control,
initiative, working memory, planning, task monitoring, organization), WAIS-IV 4
cognitive indices (verbal comprehension, perceptual reasoning, working memory,
processing speed), ADHD-RS 2 dimensions (inattention, hyperactivity), and LSAS
social anxiety subscales (anxiety, avoidance). This is the largest block because
autism is a heterogeneous condition requiring multi-dimensional characterization.

---

For each block below, violin plots show the distribution of every continuous/ordinal
feature, faceted by cohort. Inspect for:
- **Floor/ceiling effects**: thick spike at min/max → low discriminative power
- **Multimodality**: distinct humps → natural subgroups (good for stratification)
- **Extreme skew**: long thin tail → may distort cosine/euclidean distance

In [7]:
ALL_BLOCKS = [b.id for b in schema.blocks]

for block_id in ALL_BLOCKS:
    block_feats = [f.id for f in schema.features if f.block == block_id and f.id in ds.X.columns]
    # Determine which cohorts contribute to this block
    block_cohorts = set()
    for f in schema.features:
        if f.block == block_id:
            block_cohorts.update(f.cohorts)

    # Only keep continuous/ordinal features (violins don't make sense for binary)
    continuous_feats = [
        fid for fid in block_feats
        if ds.feature_metadata.loc[fid, 'type'] in ('continuous', 'ordinal')
    ]
    if not continuous_feats:
        print(f'{block_id}: no continuous features, skipping violin')
        continue

    # Build long-form dataframe
    rows = []
    for feat in continuous_feats:
        # Shorten label: strip common prefixes
        short = feat
        for pfx in ('inst_', 'asp_', 'sz_', 'dr_', 'cog_', 'np_', 'bio_', 'tx_',
                     'sub_', 'fh_', 'cm_', 'sui_', 'psyh_', 'demo_'):
            if short.startswith(pfx):
                short = short[len(pfx):]
                break
        for c in sorted(block_cohorts):
            mask = cohort == c
            vals = ds.X.loc[mask, feat].dropna()
            for v in vals:
                rows.append({'feature': short, 'feature_id': feat,
                             'cohort': c.upper(), 'value': v})

    if not rows:
        print(f'{block_id}: no data')
        continue

    df_long = pd.DataFrame(rows)
    n_feats = df_long['feature'].nunique()

    fig = px.violin(
        df_long, x='feature', y='value', color='cohort',
        title=f'Block: {block_id} — feature distributions ({n_feats} features)',
        color_discrete_map={c.upper(): COHORT_COLORS[c] for c in COHORT_COLORS},
        template='plotly_dark',
    )
    w = max(600, n_feats * 80)
    fig.update_layout(
        **DARK,
        height=450, width=w,
        xaxis_tickangle=-45,
        xaxis_tickfont=dict(size=8),
        showlegend=True,
    )
    fig.write_image(str(OUT / f'fig02_{block_id}_distributions.png'),
                    width=w, height=450, scale=SCALE)
    fig.show()

## 4. Inter-feature correlation heatmaps — all blocks (redundancy check)

Within each block, compute Spearman rank correlation matrix (pairwise-complete).
Blocks with fewer than 3 continuous features are skipped (correlation matrix not
meaningful). Identifies highly correlated subscales (|r| > 0.85) that may be
redundant — important for deciding whether to keep both a total score and its
constituent subscales, or whether two instruments measure the same construct.

In [8]:
all_high_corr_pairs = []  # Collect across all blocks for a summary table

for block_id in ALL_BLOCKS:
    block_feats = [f.id for f in schema.features if f.block == block_id and f.id in ds.X.columns]
    # Only continuous/ordinal — binary features produce degenerate correlations
    block_feats = [fid for fid in block_feats
                   if ds.feature_metadata.loc[fid, 'type'] in ('continuous', 'ordinal')]
    if len(block_feats) < 3:
        print(f'{block_id}: {len(block_feats)} continuous features, skipping correlation')
        continue

    X_block = ds.X[block_feats]
    valid_mask = X_block.notna().sum(axis=1) >= 2
    if valid_mask.sum() < 10:
        print(f'{block_id}: only {valid_mask.sum()} valid rows, skipping')
        continue
    corr = X_block.loc[valid_mask].corr(method='spearman')

    # Short labels
    short_labels = []
    for f in block_feats:
        s = f
        for pfx in ('inst_', 'asp_', 'sz_', 'dr_', 'cog_', 'np_', 'bio_', 'tx_',
                     'sub_', 'fh_', 'cm_', 'sui_', 'psyh_', 'demo_'):
            if s.startswith(pfx):
                s = s[len(pfx):]
                break
        short_labels.append(s)

    sz = max(350, len(block_feats) * 30)
    fig = go.Figure(data=go.Heatmap(
        z=corr.values,
        x=short_labels, y=short_labels,
        colorscale='RdBu_r', zmid=0, zmin=-1, zmax=1,
        colorbar=dict(title='Spearman r'),
        hovertemplate='%{x} vs %{y}<br>r = %{z:.2f}<extra></extra>',
    ))
    fig.update_layout(
        title=f'{block_id} — inter-feature Spearman correlation ({len(block_feats)} features)',
        height=sz, width=sz + 150,
        xaxis_tickangle=-45,
        xaxis_tickfont=dict(size=8),
        yaxis_tickfont=dict(size=8),
        **DARK,
    )
    fig.write_image(str(OUT / f'fig03_{block_id}_correlation.png'),
                    width=sz + 150, height=sz, scale=SCALE)
    fig.show()

    # Flag high correlations
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    high_corr = upper.stack()
    high_corr = high_corr[high_corr.abs() > 0.85].sort_values(ascending=False)
    if len(high_corr):
        print(f'\n{block_id} — high correlations (|r| > 0.85):')
        for (f1, f2), r in high_corr.items():
            print(f'  {f1} × {f2}: r = {r:.3f}')
            all_high_corr_pairs.append({'block': block_id, 'feat_1': f1, 'feat_2': f2, 'r': r})
    else:
        print(f'\n{block_id}: no correlations > 0.85')

# Summary of all high-correlation pairs across blocks
if all_high_corr_pairs:
    hc_df = pd.DataFrame(all_high_corr_pairs).sort_values('r', ascending=False)
    print(f'\n{"=" * 70}')
    print(f'TOTAL high-correlation pairs (|r| > 0.85) across all blocks: {len(hc_df)}')
    print(hc_df.to_string(index=False))
else:
    print('\nNo high-correlation pairs found in any block.')

demographics: 2 continuous features, skipping correlation



mood: no correlations > 0.85



psychosis — high correlations (|r| > 0.85):
  inst_panss_n × inst_panss_wallwork_negative: r = 0.955
  inst_panss_total × inst_panss_g: r = 0.904



anxiety_impulsivity: no correlations > 0.85



functioning: no correlations > 0.85


...


autism_profile — high correlations (|r| > 0.85):
  asp_rbs_r_total × asp_rbs_r_sameness: r = 0.917
  asp_lsas_anxiety × asp_lsas_avoidance: r = 0.884
  asp_rbs_r_total × asp_rbs_r_stereotypies: r = 0.879
  asp_adhd_rs_inattention × asp_brief_planning: r = 0.870
  asp_rbs_r_total × asp_brief_self_control: r = 0.866
  asp_rbs_r_stereotypies × asp_brief_self_control: r = 0.866
  asp_rbs_r_self_injury × asp_brief_self_control: r = 0.866
  asp_rbs_r_compulsive × asp_brief_self_control: r = 0.866
  asp_rbs_r_rituals × asp_brief_self_control: r = 0.866
  asp_rbs_r_sameness × asp_brief_self_control: r = 0.866
  asp_rbs_r_restricted × asp_brief_self_control: r = 0.866
  asp_rbs_r_total × asp_rbs_r_compulsive: r = 0.859
  asp_rbs_r_stereotypies × asp_rbs_r_rituals: r = 0.856



hostility_aggression — high correlations (|r| > 0.85):
  inst_bdhi_resentment × inst_bdhi_attitudinal: r = 0.894
  inst_bdhi_suspicion × inst_bdhi_attitudinal: r = 0.887



treatment_resistance: no correlations > 0.85



neuropsych — high correlations (|r| > 0.85):
  np_digit_span_backward_std × np_digit_span_total_std: r = 0.852



personality: no correlations > 0.85

TOTAL high-correlation pairs (|r| > 0.85) across all blocks: 23
               block                     feat_1                       feat_2        r
           psychosis               inst_panss_n inst_panss_wallwork_negative 0.955217
 psychiatric_history     psyh_age_first_episode          psyh_onset_category 0.937457
             biology               bio_waist_cm       bio_waist_height_ratio 0.934630
           cognition   cog_cvlt_long_delay_free         cog_cvlt_recognition 0.933200
      autism_profile            asp_rbs_r_total           asp_rbs_r_sameness 0.917374
           psychosis           inst_panss_total                 inst_panss_g 0.903725
hostility_aggression       inst_bdhi_resentment        inst_bdhi_attitudinal 0.893510
hostility_aggression        inst_bdhi_suspicion        inst_bdhi_attitudinal 0.886701
           cognition          cog_tmt_b_seconds            cog_tmt_b_minus_a 0.886219
      autism_profile           asp_lsa

## 5. Trans-cohort feature selection analysis

Uses `select_transdiagnostic_features()` to identify the feature set used for the
cross-cohort similarity graph. A feature is "trans-cohort" if it has ≥50% observed
coverage in every cohort. Shows which features pass the threshold and which are excluded.

In [9]:
from face_stratification.graph.transdiagnostic import select_transdiagnostic_features

fs = select_transdiagnostic_features(ds.X, ds.metadata, ds.schema)
print(f'Trans-cohort features selected: {fs.n_selected} / {ds.n_features}')
print(f'Excluded by coverage: {len(fs.excluded_by_coverage)}')
print(f'Excluded by config: {len(fs.excluded_by_config)}')
print()
print('Selected trans-cohort features:')
for fid in fs.feature_ids:
    block = ds.feature_metadata.loc[fid, 'block'] if fid in ds.feature_metadata.index else '?'
    covs = ', '.join(f'{c}={fs.per_cohort_coverage.loc[fid, c]:.0%}' for c in ('bp', 'sz', 'asp'))
    print(f'  [{block:20s}] {fid:40s} {covs}')

Trans-cohort features selected: 9 / 184
Excluded by coverage: 175
Excluded by config: 0

Selected trans-cohort features:
  [demographics        ] demo_age_years                           bp=100%, sz=100%, dr=100%, asp=100%
  [demographics        ] demo_sex_male                            bp=100%, sz=100%, dr=100%, asp=100%
  [substance           ] sub_tobacco_current                      bp=100%, sz=100%, dr=100%, asp=100%
  [substance           ] sub_alcohol_current                      bp=100%, sz=100%, dr=100%, asp=100%
  [substance           ] sub_cannabis_current                     bp=100%, sz=100%, dr=100%, asp=100%
  [substance           ] sub_use_disorder                         bp=100%, sz=100%, dr=100%, asp=100%
  [comorbidities       ] cm_n_somatic                             bp=100%, sz=100%, dr=100%, asp=100%
  [comorbidities       ] cm_n_psychiatric                         bp=100%, sz=100%, dr=100%, asp=100%
  [treatment           ] tx_polypharmacy_index                 

In [10]:
# Bar chart: per-block count of trans-cohort vs total features
block_counts = []
for b in schema.blocks:
    total = sum(1 for f in schema.features if f.block == b.id)
    n_tc = sum(1 for fid in fs.feature_ids if ds.feature_metadata.loc[fid, 'block'] == b.id)
    block_counts.append({'block': b.id, 'total': total, 'trans_cohort': n_tc})
bc = pd.DataFrame(block_counts).set_index('block')
bc = bc[bc['total'] > 0].sort_values('total', ascending=True)

fig = go.Figure()
fig.add_trace(go.Bar(y=bc.index, x=bc['total'], name='Total features',
                     orientation='h', marker_color='#4a5568'))
fig.add_trace(go.Bar(y=bc.index, x=bc['trans_cohort'], name='Trans-cohort',
                     orientation='h', marker_color='#48bb78'))
fig.update_layout(
    title='Features per block: total vs trans-cohort',
    barmode='overlay',
    height=500, width=800,
    xaxis_title='Number of features',
    **DARK,
)
fig.write_image(str(OUT / 'fig04_trans_cohort_by_block.png'), width=800, height=500, scale=SCALE)
fig.show()

## 6. Missingness pattern analysis

For each block, characterize the missingness mechanism (MCAR/MAR/MNAR) and plot a
global missingness correlation heatmap across **all** features. This reveals whether
features go missing together (instrument-level dropout) or independently.

In [11]:
from face_stratification.harmonization.missingness import characterize_missingness
miss_info = characterize_missingness(ds.X, ds.metadata, ds.schema)

# Missingness mechanism per block
print('Missingness mechanisms per block:')
for block_id, mechanism in sorted(miss_info.get('mechanism_summary', {}).items()):
    test = miss_info.get('mcar_test_per_block', {}).get(block_id, (None, None, None))
    p_str = f'p={test[1]:.4f}' if test[1] is not None else 'p=N/A'
    print(f'  {block_id:25s} {mechanism:5s} ({p_str})')

Missingness mechanisms per block:
  anxiety_impulsivity       MCAR  (p=1.0000)
  autism_profile            MCAR  (p=1.0000)
  biology                   MCAR  (p=1.0000)
  cognition                 MCAR  (p=1.0000)
  comorbidities             MCAR  (p=1.0000)
  demographics              MCAR  (p=0.3912)
  family_history            MAR   (p=0.0000)
  functioning               MCAR  (p=1.0000)
  hostility_aggression      MAR   (p=0.0000)
  insight                   MAR   (p=0.0000)
  mood                      MCAR  (p=1.0000)
  neuropsych                MCAR  (p=1.0000)
  personality               MNAR  (p=0.0000)
  psychiatric_history       MAR   (p=0.0000)
  psychosis                 MAR   (p=0.0000)
  sleep_circadian           MAR   (p=0.0000)
  substance                 MAR   (p=0.0001)
  suicide_history           MAR   (p=0.0000)
  trauma                    MAR   (p=0.0000)
  treatment                 MCAR  (p=1.0000)
  treatment_resistance      MCAR  (p=1.0000)


In [12]:
# Per-block missingness correlation heatmaps
for block_id in ALL_BLOCKS:
    block_feats = [f.id for f in schema.features if f.block == block_id and f.id in ds.X.columns]
    if len(block_feats) < 3:
        continue

    miss_indicators = ds.X[block_feats].isna().astype(float)
    # Skip blocks where missingness is constant (all present or all missing per feature)
    var = miss_indicators.var()
    variable_feats = var[var > 0].index.tolist()
    if len(variable_feats) < 3:
        continue

    miss_corr = miss_indicators[variable_feats].corr()
    short_labels = []
    for f in variable_feats:
        s = f
        for pfx in ('inst_', 'asp_', 'sz_', 'dr_', 'cog_', 'np_', 'bio_', 'tx_',
                     'sub_', 'fh_', 'cm_', 'sui_', 'psyh_', 'demo_'):
            if s.startswith(pfx):
                s = s[len(pfx):]
                break
        short_labels.append(s)

    sz = max(350, len(variable_feats) * 22)
    fig = go.Figure(data=go.Heatmap(
        z=miss_corr.values,
        x=short_labels, y=short_labels,
        colorscale='Reds', zmin=0, zmax=1,
        colorbar=dict(title='Miss. corr.'),
    ))
    fig.update_layout(
        title=f'{block_id} — missingness correlation ({len(variable_feats)} features)',
        height=sz, width=sz + 150,
        xaxis_tickangle=-45,
        xaxis_tickfont=dict(size=7),
        yaxis_tickfont=dict(size=7),
        **DARK,
    )
    fig.write_image(str(OUT / f'fig05_{block_id}_missingness.png'),
                    width=sz + 150, height=sz, scale=SCALE)
    fig.show()

## 7. t-SNE embedding colored by new features

PCA → t-SNE projection on the trans-cohort feature subset, colored by:
1. Cohort membership
2. Polypharmacy index (new trans-cohort feature)
3. CTQ emotional abuse (now correctly extracted)
4. BMI

In [13]:
from face_stratification.harmonization.normalization import fit_normalization, transform_normalization
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Normalize
stats = fit_normalization(ds.X, ds.schema)
X_norm = transform_normalization(ds.X, stats)

# Use trans-cohort features for projection
X_td = X_norm[list(fs.feature_ids)]

# Fill NaN with 0 for PCA/t-SNE (required)
X_filled = X_td.fillna(0)

# PCA → 20 components → t-SNE
n_comp = min(20, X_filled.shape[1])
pca_coords = PCA(n_components=n_comp, random_state=0).fit_transform(X_filled)
tsne_coords = TSNE(n_components=2, perplexity=30, random_state=0, init='pca',
                   learning_rate='auto').fit_transform(pca_coords)

proj = pd.DataFrame(tsne_coords, columns=['t-SNE 1', 't-SNE 2'], index=ds.X.index)
proj['cohort'] = cohort.values
proj['cohort_name'] = proj['cohort'].map(COHORT_NAMES)
print(f't-SNE done: {proj.shape[0]} patients, {n_comp} PCA components')

t-SNE done: 1200 patients, 9 PCA components


In [14]:
# Panel 1: colored by cohort
fig = px.scatter(
    proj, x='t-SNE 1', y='t-SNE 2', color='cohort_name',
    color_discrete_map={COHORT_NAMES[c]: COHORT_COLORS[c] for c in COHORT_COLORS},
    title='t-SNE projection — colored by cohort',
    opacity=0.6, template='plotly_dark',
)
fig.update_layout(**DARK, height=550, width=700)
fig.update_traces(marker_size=4)
fig.write_image(str(OUT / 'fig06_tsne_cohort.png'), width=700, height=550, scale=SCALE)
fig.show()

In [15]:
# Panel 2: colored by new derived / fixed features
color_features = [
    ('tx_polypharmacy_index', 'Polypharmacy index'),
    ('inst_ctq_emotional_abuse', 'CTQ emotional abuse'),
    ('inst_ctq_total', 'CTQ total'),
    ('bio_bmi', 'BMI'),
]

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[label for _, label in color_features],
    horizontal_spacing=0.08, vertical_spacing=0.12,
)

for idx, (feat_id, label) in enumerate(color_features):
    row, col = divmod(idx, 2)
    vals = ds.X[feat_id].reindex(proj.index) if feat_id in ds.X.columns else pd.Series(np.nan, index=proj.index)
    
    fig.add_trace(go.Scatter(
        x=proj['t-SNE 1'], y=proj['t-SNE 2'],
        mode='markers',
        marker=dict(
            size=3, opacity=0.6,
            color=vals.values,
            colorscale='Turbo',
            colorbar=dict(title=label, len=0.4, y=0.8 - row * 0.5, x=1.0 + col * 0.08) if col == 1 else dict(title=label, len=0.4, y=0.8 - row * 0.5),
            showscale=(col == 1),
        ),
        showlegend=False,
        hovertemplate=f'{label}: ' + '%{marker.color:.1f}<extra></extra>',
    ), row=row + 1, col=col + 1)

fig.update_layout(
    title='t-SNE colored by clinical features',
    height=800, width=1000,
    **DARK,
)
fig.write_image(str(OUT / 'fig07_tsne_features.png'), width=1000, height=800, scale=SCALE)
fig.show()

## 8. Block-level feature count and coverage summary

Final summary tables: per-block feature count, per-cohort emission count, and
comparison with the pre-expansion baseline.

In [16]:
# Per-block summary
block_summary = []
for b in schema.blocks:
    feats = [f for f in schema.features if f.block == b.id]
    n_total = len(feats)
    n_trans_cohort = sum(1 for f in feats if len(f.cohorts) >= 3)
    # Mean coverage across the features in this block
    feat_ids = [f.id for f in feats if f.id in ds.X.columns]
    mean_cov = ds.X[feat_ids].notna().mean().mean() if feat_ids else 0
    block_summary.append({
        'block': b.id,
        'n_features': n_total,
        'n_all_cohort': n_trans_cohort,
        'metric': b.metric,
        'mean_coverage': f'{mean_cov:.0%}',
    })

summary_df = pd.DataFrame(block_summary)
print(summary_df.to_string(index=False))
print(f'\nTotal features: {summary_df["n_features"].sum()}')

               block  n_features  n_all_cohort    metric mean_coverage
        demographics           5             5     gower           64%
                mood           9             0    cosine           37%
           psychosis          11             0    cosine           24%
 anxiety_impulsivity           8             0    cosine           23%
         functioning           4             0    cosine           29%
     sleep_circadian           3             1    cosine           45%
           cognition          17             0 euclidean           20%
             biology          13             1 euclidean           29%
           treatment          11             4    cosine           44%
           substance           5             5     gower           80%
              trauma           7             6    cosine           59%
      family_history           4             4     gower           75%
       comorbidities           2             2     gower          100%
     s

In [17]:
# Per-cohort feature emission count
print('Per-cohort features emitted (non-NaN for ≥1 patient):')
for c in ('bp', 'sz', 'asp'):
    mask = cohort == c
    n_emitted = (ds.X.loc[mask].notna().any()).sum()
    print(f'  {c}: {n_emitted} / {ds.n_features} features with any data')

Per-cohort features emitted (non-NaN for ≥1 patient):
  bp: 96 / 184 features with any data
  sz: 71 / 184 features with any data
  dr: 68 / 184 features with any data
  asp: 53 / 184 features with any data


In [18]:
# Stacked bar chart: feature count per block colored by block
fig = px.bar(
    summary_df.sort_values('n_features', ascending=True),
    y='block', x='n_features', orientation='h',
    title='Feature count per block',
    color='block',
    template='plotly_dark',
    text='n_features',
)
fig.update_layout(
    **DARK, height=500, width=800,
    showlegend=False,
    xaxis_title='Number of features',
)
fig.write_image(str(OUT / 'fig08_features_per_block.png'), width=800, height=500, scale=SCALE)
fig.show()

## 9. Derived feature validation

Quick sanity checks on the computed derived features:
- Polypharmacy distribution per cohort
- Onset category distribution
- Illness burden distribution
- Waist/height ratio distribution

In [19]:
derived_feats = ['tx_polypharmacy_index', 'psyh_onset_category', 'psyh_illness_burden', 'bio_waist_height_ratio']

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f.replace('tx_', '').replace('psyh_', '').replace('bio_', '') for f in derived_feats],
)

for idx, feat in enumerate(derived_feats):
    row, col = divmod(idx, 2)
    for c in ('bp', 'sz', 'asp'):
        mask = cohort == c
        vals = ds.X.loc[mask, feat].dropna() if feat in ds.X.columns else pd.Series(dtype=float)
        if len(vals) == 0:
            continue
        fig.add_trace(go.Histogram(
            x=vals, name=c.upper(),
            marker_color=COHORT_COLORS[c],
            opacity=0.6,
            showlegend=(idx == 0),
        ), row=row + 1, col=col + 1)

fig.update_layout(
    title='Derived feature distributions per cohort',
    barmode='overlay',
    height=600, width=1000,
    **DARK,
)
fig.write_image(str(OUT / 'fig09_derived_distributions.png'), width=1000, height=600, scale=SCALE)
fig.show()

## 10. PANSS Wallwork vs P/N/G comparison (SZ only)

The Wallwork 5-factor model should show differential structure compared to the
traditional 3-subscale model. Scatter matrix and correlation.

In [20]:
panss_feats = [
    'inst_panss_p', 'inst_panss_n', 'inst_panss_g',
    'inst_panss_wallwork_positive', 'inst_panss_wallwork_negative',
    'inst_panss_wallwork_disorganized', 'inst_panss_wallwork_excited',
    'inst_panss_wallwork_depressed',
]
panss_feats = [f for f in panss_feats if f in ds.X.columns]

sz_mask = cohort == 'sz'
panss_df = ds.X.loc[sz_mask, panss_feats].dropna()

if len(panss_df) >= 10:
    corr = panss_df.corr(method='spearman')
    short = [f.replace('inst_panss_', '').replace('wallwork_', 'W:') for f in panss_feats]
    
    fig = go.Figure(data=go.Heatmap(
        z=corr.values, x=short, y=short,
        colorscale='RdBu_r', zmid=0, zmin=-1, zmax=1,
        colorbar=dict(title='Spearman r'),
        text=corr.round(2).values, texttemplate='%{text}',
    ))
    fig.update_layout(
        title='PANSS: Traditional (P/N/G) vs Wallwork 5-factor correlation',
        height=450, width=550,
        **DARK,
    )
    fig.write_image(str(OUT / 'fig10_panss_wallwork_vs_png.png'), width=550, height=450, scale=SCALE)
    fig.show()
else:
    print(f'Only {len(panss_df)} SZ patients with complete PANSS data')

## 11. Subscale fix validation — before/after CTQ and BIS-10

The French key fix in Phase 2 changed CTQ and BIS-10 subscales from all-None to real values.
Verify that the subscale totals sum approximately to the instrument total.

In [21]:
# CTQ: total should ≈ sum of 5 subscales (they don't perfectly sum, but should correlate)
ctq_sub = ['inst_ctq_emotional_abuse', 'inst_ctq_physical_abuse', 'inst_ctq_sexual_abuse',
           'inst_ctq_emotional_neglect', 'inst_ctq_physical_neglect']
ctq_cols = ['inst_ctq_total'] + ctq_sub
ctq_cols = [c for c in ctq_cols if c in ds.X.columns]
ctq_data = ds.X[ctq_cols].dropna()

if len(ctq_data) > 10:
    ctq_data['sum_subscales'] = ctq_data[ctq_sub].sum(axis=1)
    r = ctq_data['inst_ctq_total'].corr(ctq_data['sum_subscales'])
    print(f'CTQ total vs sum(subscales): r = {r:.3f} (n={len(ctq_data)})')
    print(f'  Mean total: {ctq_data["inst_ctq_total"].mean():.1f}')
    print(f'  Mean sum:   {ctq_data["sum_subscales"].mean():.1f}')
else:
    print(f'Only {len(ctq_data)} patients with complete CTQ data')

# BIS-10: total should ≈ sum of 3 subscales
bis_sub = ['inst_bis10_attentional', 'inst_bis10_motor', 'inst_bis10_nonplanning']
bis_cols = ['inst_bis10_total'] + bis_sub
bis_cols = [c for c in bis_cols if c in ds.X.columns]
bis_data = ds.X[bis_cols].dropna()

if len(bis_data) > 10:
    bis_data['sum_subscales'] = bis_data[bis_sub].sum(axis=1)
    r = bis_data['inst_bis10_total'].corr(bis_data['sum_subscales'])
    print(f'\nBIS-10 total vs sum(subscales): r = {r:.3f} (n={len(bis_data)})')
    print(f'  Mean total: {bis_data["inst_bis10_total"].mean():.1f}')
    print(f'  Mean sum:   {bis_data["sum_subscales"].mean():.1f}')
else:
    print(f'Only {len(bis_data)} patients with complete BIS-10 data')

CTQ total vs sum(subscales): r = 1.000 (n=815)
  Mean total: 42.9
  Mean sum:   42.9

BIS-10 total vs sum(subscales): r = 1.000 (n=280)
  Mean total: 77.5
  Mean sum:   77.5


## 12. Summary statistics table

Final overview of all features with descriptive statistics per cohort.

In [22]:
# Build summary: for each feature, show mean/std/coverage per cohort
summary_rows = []
for feat in ds.feature_metadata.index:
    if feat not in ds.X.columns:
        continue
    row = {
        'feature': feat,
        'block': ds.feature_metadata.loc[feat, 'block'],
        'type': ds.feature_metadata.loc[feat, 'type'],
    }
    for c in ('bp', 'sz', 'asp'):
        vals = ds.X.loc[cohort == c, feat]
        cov = vals.notna().mean()
        if cov > 0:
            row[f'{c}_mean'] = f'{vals.mean():.1f}'
            row[f'{c}_std'] = f'{vals.std():.1f}'
            row[f'{c}_cov'] = f'{cov:.0%}'
        else:
            row[f'{c}_mean'] = '-'
            row[f'{c}_std'] = '-'
            row[f'{c}_cov'] = '-'
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows).set_index('feature')
print(f'Complete feature summary: {len(summary)} features')
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', None):
    display(summary.style.set_properties(**{'text-align': 'right', 'font-size': '11px'})
            .set_table_styles([
                {'selector': 'th', 'props': [('font-size', '11px'), ('text-align', 'center')]},
                {'selector': 'td', 'props': [('white-space', 'nowrap')]},
            ]))

Complete feature summary: 184 features


,block,type,bp_mean,bp_std,bp_cov,sz_mean,sz_std,sz_cov,dr_mean,dr_std,dr_cov,asp_mean,asp_std,asp_cov
feature,,,,,,,,,,,,,,
demo_age_years,demographics,continuous,38.8,13.6,100%,32.0,10.0,100%,50.1,13.1,100%,28.4,11.1,100%
demo_sex_male,demographics,binary,0.4,0.5,100%,0.7,0.4,100%,0.4,0.5,100%,0.5,0.5,100%
demo_education_years_ordinal,demographics,ordinal,4.5,1.0,68%,4.0,1.0,83%,4.1,1.5,67%,1.1,0.2,23%
demo_marital_partnered,demographics,binary,0.4,0.5,78%,0.0,0.0,84%,0.6,0.5,77%,-,-,-
demo_employed,demographics,binary,-,-,-,-,-,-,1.0,nan,0%,-,-,-
inst_madrs_total,mood,continuous,9.3,8.1,97%,-,-,-,28.1,7.5,97%,-,-,-
inst_ymrs_total,mood,continuous,2.0,3.1,96%,1.3,2.6,97%,1.2,2.0,66%,-,-,-
inst_cgis_total,mood,ordinal,3.4,1.4,80%,4.3,1.2,98%,4.9,0.9,92%,0.3,0.5,3%
inst_qids_total,mood,continuous,9.2,6.0,93%,-,-,-,17.1,4.6,94%,-,-,-


In [23]:
# Export summary to CSV
summary.to_csv(OUT / 'feature_summary.csv')
print(f'Saved to {OUT / "feature_summary.csv"}')
print(f'\n=== DIAGNOSTIC COMPLETE ===')
print(f'Schema: {schema.version}, {len(schema.blocks)} blocks, {len(schema.features)} features')
print(f'Harmonized: {ds.n_patients} patients, {ds.n_features} features')
print(f'Trans-cohort: {fs.n_selected} features')
print(f'Figures saved to: {OUT}')

Saved to /Users/andriikulakovskyi/Desktop/llm-rl/psych-dataset/output/stratification/stage_a_inspection/feature_summary.csv

=== DIAGNOSTIC COMPLETE ===
Schema: 0.2.0, 21 blocks, 184 features
Harmonized: 1200 patients, 184 features
Trans-cohort: 9 features
Figures saved to: /Users/andriikulakovskyi/Desktop/llm-rl/psych-dataset/output/stratification/stage_a_inspection


## 13. Graph Construction and Topology Analysis

This section builds the multiplex patient graph from the harmonized feature matrix, computes graph-level statistics, and analyses the resulting topology. The graph is the **contract between Stage A and Stage B** — all downstream embedding and clustering consume this graph, never the raw feature matrix.

The analysis covers:
1. **Graph construction** — Build 21 block-level kNN graphs + 1 transdiagnostic layer, assemble into a multiplex, apply per-block weight normalization.
2. **Empirical statistics** — Edge counts, mean degree, cohort assortativity, candidate coverage per block.
3. **Assortativity analysis** — Which blocks bridge across cohorts vs remain cohort-specific?
4. **Cross-cohort bridge topology** — Which clinical domains connect which cohort pairs?
5. **Weight normalization** — Eliminates the 490× imbalance between low- and high-dimensional blocks.
6. **Deep topology analysis** — Block taxonomy, transdiagnosticity spectrum, bridge composition, per-patient connectivity.
7. **Conclusions** — Data science and clinical interpretation of the graph structure.

In [24]:
# Build the multiplex graph from the harmonized dataset
import networkx as nx
from face_stratification.graph.patient_similarity import (
    build_multiplex_graph,
    summarize_graph,
)

G, built, td_result = build_multiplex_graph(
    ds.X,
    schema,
    k=10,
    metadata=ds.metadata,
    normalize_block_weights=True,
)
gs = summarize_graph(G, schema, built, td_result)

print(f'Nodes: {gs.n_nodes:,}')
print(f'Edge types: {gs.n_edge_types}')
print(f'Total edges: {sum(gs.edges_per_type.values()):,}')
print(f'Dead blocks: {gs.blocks_with_zero_edges}')
print()
print(f'{"Block":30s} {"Edges":>8s} {"Degree":>8s} {"Assort":>8s} {"Cands":>8s}')
print('-' * 72)
for bid in sorted(gs.edges_per_type, key=lambda b: gs.edges_per_type[b], reverse=True):
    e = gs.edges_per_type[bid]
    d = gs.mean_degree_per_type.get(bid, 0)
    a = gs.cohort_assortativity.get(bid, float('nan'))
    c = gs.candidate_nodes_per_type.get(bid, 0)
    print(f'{bid:30s} {e:8,d} {d:8.1f} {a:+8.3f} {c:8,d}')

Nodes: 1,200
Edge types: 22
Total edges: 103,035
Dead blocks: []

Block                             Edges   Degree   Assort    Cands
------------------------------------------------------------------------
substance                        11,890     19.8   -0.177    1,200
comorbidities                    11,412     19.0   +0.083    1,200
functioning                       9,254     15.4   +0.088      937
transdiagnostic                   8,860     14.8   +0.387    1,200
family_history                    8,481     14.1   +0.114      900
suicide_history                   7,154     11.9   +0.283      758
trauma                            5,482      9.1   +0.038      815
demographics                      5,197      8.7   +0.380      821
neuropsych                        4,252      7.1   -0.472      508
treatment                         4,167      6.9   +0.440      570
sleep_circadian                   3,647      6.1   +0.039      502
anxiety_impulsivity               3,508      5.8   +0.209

/opt/anaconda3/lib/python3.13/site-packages/networkx/algorithms/assortativity/correlation.py:282: RuntimeWarning:

invalid value encountered in scalar divide



### 13.1 Empirical graph statistics table

Sorted by edge count. Key columns:
- **Assortativity**: Newman's cohort-label assortativity. +1 = only intra-cohort edges, −1 = preferentially cross-cohort, 0 = random mixing.
- **Candidates**: number of patients participating in the block's graph (those passing `min_fraction_present`).
- **Bridging?**: which cohort pairs are connected (assortativity < 1.0 implies cross-cohort edges).

In [25]:
# Build a DataFrame with graph statistics
rows = []
for bid in sorted(gs.edges_per_type, key=lambda b: gs.edges_per_type[b], reverse=True):
    block_cohorts = set()
    for f in schema.features:
        if f.block == bid:
            block_cohorts.update(f.cohorts)
    if bid == 'transdiagnostic':
        block_cohorts = {'bp', 'sz', 'asp'}
    
    a = gs.cohort_assortativity.get(bid, float('nan'))
    bridging = 'All 3 cohorts' if len(block_cohorts) >= 3 else (
        ' + '.join(sorted(c.upper() for c in block_cohorts)) if a >= 0.999 else
        f'{" + ".join(sorted(c.upper() for c in block_cohorts))} (bridging)'
    )
    rows.append({
        'Block': bid,
        'Edges': gs.edges_per_type[bid],
        'Mean Degree': round(gs.mean_degree_per_type.get(bid, 0), 1),
        'Assortativity': round(a, 3),
        'Candidates': gs.candidate_nodes_per_type.get(bid, 0),
        'Bridging': bridging,
    })

graph_stats = pd.DataFrame(rows)

# Color-code assortativity
def assort_color(val):
    if val >= 0.999:
        return '#ef4444'  # red — cohort-specific
    elif val > 0.3:
        return '#fbbf24'  # amber — weakly bridging
    elif val > -0.1:
        return '#34d399'  # green — well-mixed
    else:
        return '#6366f1'  # indigo — strongly cross-cohort

colors = [assort_color(r['Assortativity']) for _, r in graph_stats.iterrows()]

fig = go.Figure(data=[go.Table(
    header=dict(
        values=['Block', 'Edges', 'Mean Degree', 'Assortativity', 'Candidates', 'Bridging'],
        fill_color='#1a1d27',
        font=dict(color='white', size=12),
        align='left',
    ),
    cells=dict(
        values=[graph_stats[c] for c in graph_stats.columns],
        fill_color=[
            ['#0f1117'] * len(graph_stats),  # Block
            ['#0f1117'] * len(graph_stats),  # Edges
            ['#0f1117'] * len(graph_stats),  # Mean Degree
            colors,                           # Assortativity — color-coded
            ['#0f1117'] * len(graph_stats),  # Candidates
            ['#0f1117'] * len(graph_stats),  # Bridging
        ],
        font=dict(color='white', size=11),
        align='left',
    ),
)])
fig.update_layout(
    title='Multiplex graph — empirical statistics per block',
    height=700, width=1100,
    **DARK,
)
fig.write_image(str(OUT / 'graph_10_statistics_table.png'), width=1100, height=700, scale=SCALE)
fig.show()

### 13.2 Cohort assortativity per block

Assortativity measures whether edges connect same-cohort or cross-cohort patient pairs.
- **Negative** (indigo): patients preferentially connect across cohorts — the block captures a genuinely transdiagnostic dimension.
- **Near zero** (green): random mixing — the block does not discriminate by diagnosis.
- **Positive** (amber): moderate within-cohort preference.
- **+1.0** (red): single-cohort block — all edges are within one diagnostic group by construction.

In [26]:
# Assortativity bar chart
assort_df = graph_stats.sort_values('Assortativity')

colors_bar = []
for a in assort_df['Assortativity']:
    if a >= 0.999:
        colors_bar.append('#ef4444')
    elif a > 0.3:
        colors_bar.append('#fbbf24')
    elif a > -0.1:
        colors_bar.append('#34d399')
    else:
        colors_bar.append('#6366f1')

fig = go.Figure(go.Bar(
    x=assort_df['Assortativity'],
    y=assort_df['Block'],
    orientation='h',
    marker_color=colors_bar,
    text=[f'{a:+.3f}' for a in assort_df['Assortativity']],
    textposition='outside',
    textfont=dict(size=10),
))
fig.add_vline(x=0, line_dash='dash', line_color='white', opacity=0.4)
fig.update_layout(
    title='Cohort assortativity per block (Newman attribute assortativity)',
    xaxis_title='Assortativity coefficient',
    yaxis_title='',
    height=600, width=900,
    xaxis=dict(range=[-1.1, 1.3]),
    yaxis=dict(tickfont=dict(size=10)),
    **DARK,
)
fig.write_image(str(OUT / 'graph_11_assortativity.png'), width=900, height=600, scale=SCALE)
fig.show()

### 13.3 Cross-cohort bridge heatmap

For each block, count the number of cross-cohort edges per cohort pair (BP-SZ, BP-ASP, SZ-ASP). Blocks with assortativity=1.0 produce zero cross-cohort edges and appear as empty rows.

In [27]:
# Count cross-cohort edges per block per cohort pair
from collections import Counter

node_cohort = nx.get_node_attributes(G, 'cohort')
cohort_pairs = ['BP-SZ', 'BP-ASP', 'SZ-ASP']

bridge_counts = {}
for u, v, data in G.edges(data=True):
    bid = data.get('block', 'unknown')
    c1, c2 = node_cohort.get(u, '?').upper(), node_cohort.get(v, '?').upper()
    if c1 != c2:
        pair = '-'.join(sorted([c1, c2]))
        bridge_counts.setdefault(bid, Counter())[pair] += 1

# Build heatmap matrix
all_blocks_sorted = graph_stats['Block'].tolist()
bridge_matrix = np.zeros((len(all_blocks_sorted), len(cohort_pairs)))
for i, bid in enumerate(all_blocks_sorted):
    for j, pair in enumerate(cohort_pairs):
        bridge_matrix[i, j] = bridge_counts.get(bid, {}).get(pair, 0)

fig = go.Figure(data=go.Heatmap(
    z=bridge_matrix,
    x=cohort_pairs,
    y=all_blocks_sorted,
    colorscale='Viridis',
    colorbar=dict(title='Cross-cohort<br>edges'),
    hovertemplate='%{y} × %{x}: %{z:,.0f} edges<extra></extra>',
    text=[[f'{int(v):,}' if v > 0 else '' for v in row] for row in bridge_matrix],
    texttemplate='%{text}',
    textfont=dict(size=9),
))
fig.update_layout(
    title='Cross-cohort edges per block × cohort pair',
    height=700, width=800,
    yaxis=dict(tickfont=dict(size=10)),
    **DARK,
)
fig.write_image(str(OUT / 'graph_12_bridge_heatmap.png'), width=800, height=700, scale=SCALE)
fig.show()

### 13.4 Per-block edge weight normalization

Without normalization, low-dimensional Gower blocks (comorbidities: 2 features) produce orders of magnitude more total edge weight than sparse clinical blocks. This causes the spectral embedding to be driven by comorbidity patterns rather than clinical profiles.

After normalization, every block contributes equal total weight. Individual weight *ratios* within each block are preserved — only the cross-block scale is equalized.

In [28]:
# Compute total weight per block (after normalization — already applied)
block_total_weight = {}
for u, v, data in G.edges(data=True):
    bid = data.get('block', 'unknown')
    block_total_weight[bid] = block_total_weight.get(bid, 0) + data.get('weight', 0)

weight_df = pd.DataFrame([
    {'Block': bid, 'Total Weight (normalized)': w}
    for bid, w in sorted(block_total_weight.items(), key=lambda x: -x[1])
])

fig = go.Figure(go.Bar(
    x=weight_df['Block'],
    y=weight_df['Total Weight (normalized)'],
    marker_color='#6366f1',
    text=[f'{w:,.0f}' for w in weight_df['Total Weight (normalized)']],
    textposition='outside',
    textfont=dict(size=8),
))
fig.update_layout(
    title='Total edge weight per block (after normalization)',
    xaxis_title='',
    yaxis_title='Total weight',
    height=500, width=1000,
    xaxis=dict(tickangle=-45, tickfont=dict(size=9)),
    **DARK,
)

# Add annotation about the normalization ratio
weights = list(block_total_weight.values())
if weights:
    ratio = max(weights) / max(min(weights), 1e-9)
    fig.add_annotation(
        text=f'Max/min ratio: {ratio:.2f}x (target: 1.0x)',
        xref='paper', yref='paper', x=0.98, y=0.95,
        showarrow=False, font=dict(color='#34d399', size=13),
    )

fig.write_image(str(OUT / 'graph_13_weight_normalized.png'), width=1000, height=500, scale=SCALE)
fig.show()

### 13.5 Cohort participation per block

Shows what fraction of each cohort's patients are candidates in each block's graph. A cell value of 1.0 means every patient in that cohort participates; 0.0 means the block's instruments were not administered to that cohort.

In [29]:
# Compute per-cohort participation rate in each block
from face_stratification.harmonization.missingness import split_blocks

per_block = split_blocks(ds.X, schema)
cohort_labels = np.array([c for c, _ in ds.X.index])
cohort_list = ['bp', 'sz', 'asp']

participation = np.zeros((len(all_blocks_sorted), len(cohort_list)))
for i, bid in enumerate(all_blocks_sorted):
    if bid == 'transdiagnostic':
        # All patients participate in transdiagnostic
        for j, c in enumerate(cohort_list):
            participation[i, j] = 1.0
        continue
    if bid not in per_block:
        continue
    block_df = per_block[bid]
    block_def = schema.block(bid)
    min_frac = block_def.min_fraction_present
    n_feats = block_df.shape[1]
    for j, c in enumerate(cohort_list):
        mask = cohort_labels == c
        if mask.sum() == 0:
            continue
        sub = block_df.loc[mask]
        frac_present = sub.notna().mean(axis=1)
        n_eligible = (frac_present >= min_frac).sum()
        participation[i, j] = n_eligible / mask.sum()

fig = go.Figure(data=go.Heatmap(
    z=participation,
    x=[c.upper() for c in cohort_list],
    y=all_blocks_sorted,
    colorscale='RdYlGn',
    zmin=0, zmax=1,
    colorbar=dict(title='Participation', tickformat='.0%'),
    hovertemplate='%{y} × %{x}: %{z:.1%}<extra></extra>',
    text=[[f'{v:.0%}' if v > 0 else '' for v in row] for row in participation],
    texttemplate='%{text}',
    textfont=dict(size=10),
))
fig.update_layout(
    title='Cohort participation rate per block (fraction of patients passing min_fraction_present)',
    height=700, width=600,
    yaxis=dict(tickfont=dict(size=10)),
    **DARK,
)
fig.write_image(str(OUT / 'graph_14_participation.png'), width=600, height=700, scale=SCALE)
fig.show()

### 13.6 Deep topology analysis — block taxonomy and transdiagnosticity spectrum

Two orthogonal dimensions define each block's transdiagnosticity:
1. **% cross-cohort edges** (quantity): what fraction of a block's edges connect patients from different cohorts?
2. **Cross/intra weight ratio** (quality): are cross-cohort edges as strong as intra-cohort ones?

A block can have many cross-cohort edges but with low weights (e.g., neuropsych: 87% cross but ratio 0.68), or fewer cross-cohort edges but with very high weights (e.g., anxiety_impulsivity: 35% cross but ratio 1.36). Both dimensions matter for the downstream embedding.

In [30]:
# Compute transdiagnosticity metrics per block
trans_rows = []
for bid in gs.edges_per_type:
    if gs.edges_per_type[bid] == 0:
        continue

    cross_edges = 0
    intra_edges = 0
    cross_weight = 0.0
    intra_weight = 0.0

    for u, v, data in G.edges(data=True):
        if data.get('block') != bid:
            continue
        w = data.get('weight', 1.0)
        if node_cohort.get(u) != node_cohort.get(v):
            cross_edges += 1
            cross_weight += w
        else:
            intra_edges += 1
            intra_weight += w

    total = cross_edges + intra_edges
    pct_cross = cross_edges / total * 100 if total > 0 else 0
    mean_cross = cross_weight / cross_edges if cross_edges > 0 else 0
    mean_intra = intra_weight / intra_edges if intra_edges > 0 else 0
    weight_ratio = mean_cross / mean_intra if mean_intra > 0 else 0

    # Classify
    a = gs.cohort_assortativity.get(bid, 1.0)
    if a >= 0.999:
        tier = 'Cohort-specific'
    elif pct_cross > 40 and weight_ratio > 0.7:
        tier = 'Strongly transdiagnostic'
    elif pct_cross > 20:
        tier = 'Selective bridge'
    else:
        tier = 'Nosology-bound'

    trans_rows.append({
        'Block': bid,
        'Cross-cohort %': round(pct_cross, 1),
        'Weight ratio': round(weight_ratio, 2),
        'Cross edges': cross_edges,
        'Intra edges': intra_edges,
        'Assortativity': round(a, 3),
        'Tier': tier,
        'Edges': total,
    })

trans_df = pd.DataFrame(trans_rows)

tier_colors = {
    'Strongly transdiagnostic': '#6366f1',
    'Selective bridge': '#34d399',
    'Nosology-bound': '#fbbf24',
    'Cohort-specific': '#ef4444',
}

fig = px.scatter(
    trans_df[trans_df['Tier'] != 'Cohort-specific'],
    x='Cross-cohort %',
    y='Weight ratio',
    color='Tier',
    size='Edges',
    text='Block',
    color_discrete_map=tier_colors,
    template='plotly_dark',
    size_max=40,
)
fig.update_traces(textposition='top center', textfont_size=9)
fig.add_hline(y=1.0, line_dash='dash', line_color='white', opacity=0.3,
              annotation_text='Equal cross/intra weight', annotation_position='top right')
fig.add_vline(x=40, line_dash='dash', line_color='white', opacity=0.3)
fig.update_layout(
    title='Transdiagnosticity spectrum — % cross-cohort edges vs cross/intra weight ratio',
    xaxis_title='Cross-cohort edges (%)',
    yaxis_title='Mean cross-cohort weight / Mean intra-cohort weight',
    height=550, width=900,
    paper_bgcolor='#0f1117', plot_bgcolor='#1a1d27',
)
fig.write_image(str(OUT / 'graph_15_transdiagnosticity.png'), width=900, height=550, scale=SCALE)
fig.show()

# Print the full table
print('\nBlock taxonomy (all blocks):')
print(trans_df.sort_values('Cross-cohort %', ascending=False).to_string(index=False))


Block taxonomy (all blocks):
               Block  Cross-cohort %  Weight ratio  Cross edges  Intra edges  Assortativity                     Tier  Edges
           cognition            80.9          0.92         2420          572         -0.645 Strongly transdiagnostic   2992
           substance            69.9          1.00         8307         3583         -0.177 Strongly transdiagnostic  11890
          neuropsych            68.9          0.92         2930         1322         -0.472 Strongly transdiagnostic   4252
              trauma            64.0          0.98         3510         1972          0.038 Strongly transdiagnostic   5482
       comorbidities            62.1          1.00         7088         4324          0.083 Strongly transdiagnostic  11412
         functioning            61.1          0.99         5658         3596          0.088 Strongly transdiagnostic   9254
      family_history            57.7          1.00         4891         3590          0.114 Strongly t

### 13.7 Bridge composition — which clinical domains connect which cohort pairs?

For each cohort pair, this stacked bar chart shows the clinical domain composition of the cross-cohort bridge. The height of each bar segment indicates the number of cross-cohort edges contributed by that clinical block.

In [31]:
# Build bridge composition per cohort pair
bridge_data = {}
for bid, counts in bridge_counts.items():
    for pair, count in counts.items():
        bridge_data.setdefault(pair, {})[bid] = count

# Get top blocks per pair for readability
cohort_pairs_sorted = sorted(bridge_data.keys(),
                             key=lambda p: sum(bridge_data[p].values()), reverse=True)

# Collect all blocks that contribute cross-cohort edges
bridge_blocks = set()
for pair_data in bridge_data.values():
    bridge_blocks.update(pair_data.keys())
bridge_blocks = sorted(bridge_blocks)

fig = go.Figure()
for bid in bridge_blocks:
    fig.add_trace(go.Bar(
        name=bid,
        x=cohort_pairs_sorted,
        y=[bridge_data.get(p, {}).get(bid, 0) for p in cohort_pairs_sorted],
    ))

fig.update_layout(
    barmode='stack',
    title='Cross-cohort bridge composition by clinical domain',
    xaxis_title='Cohort pair',
    yaxis_title='Cross-cohort edges',
    height=550, width=900,
    legend=dict(font=dict(size=9), traceorder='normal'),
    **DARK,
)
fig.write_image(str(OUT / 'graph_16_bridge_composition.png'), width=900, height=550, scale=SCALE)
fig.show()

# Print summary
print('\nCross-cohort edge counts per pair:')
for pair in cohort_pairs_sorted:
    total = sum(bridge_data[pair].values())
    top3 = sorted(bridge_data[pair].items(), key=lambda x: -x[1])[:3]
    top3_str = ', '.join(f'{b}({n:,})' for b, n in top3)
    print(f'  {pair}: {total:,} edges — top blocks: {top3_str}')


Cross-cohort edge counts per pair:
  BP-SZ: 16,177 edges — top blocks: neuropsych(2,930), cognition(2,420), suicide_history(1,679)
  BP-DR: 15,495 edges — top blocks: functioning(2,288), substance(2,260), sleep_circadian(1,750)
  DR-SZ: 9,792 edges — top blocks: substance(2,930), family_history(2,477), comorbidities(1,340)
  ASP-DR: 8,309 edges — top blocks: substance(3,000), comorbidities(2,530), transdiagnostic(1,567)
  ASP-SZ: 1,039 edges — top blocks: functioning(539), comorbidities(363), transdiagnostic(117)
  ASP-BP: 653 edges — top blocks: comorbidities(342), transdiagnostic(308), demographics(3)


### 13.8 Per-patient cross-cohort connectivity

For each patient, count the number of cross-cohort edges (neighbours from a different diagnostic group). The distribution reveals whether cross-cohort bridges are concentrated in a few "boundary" patients or spread across the cohort.

In [32]:
# Compute per-patient cross-cohort degree
cross_degree = np.zeros(gs.n_nodes)
for u, v, data in G.edges(data=True):
    if node_cohort.get(u) != node_cohort.get(v):
        cross_degree[u] += 1
        cross_degree[v] += 1

# Build DataFrame
degree_rows = []
for i in range(gs.n_nodes):
    degree_rows.append({
        'patient_idx': i,
        'cohort': node_cohort.get(i, '?').upper(),
        'cross_cohort_degree': cross_degree[i],
    })
degree_df = pd.DataFrame(degree_rows)

fig = px.violin(
    degree_df, x='cohort', y='cross_cohort_degree', color='cohort',
    color_discrete_map={c.upper(): COHORT_COLORS[c] for c in COHORT_COLORS},
    box=True, points=False,
    template='plotly_dark',
)
fig.update_layout(
    title='Per-patient cross-cohort degree (number of neighbours from other cohorts)',
    xaxis_title='Cohort',
    yaxis_title='Cross-cohort degree',
    showlegend=False,
    height=500, width=700,
    paper_bgcolor='#0f1117', plot_bgcolor='#1a1d27',
)
fig.write_image(str(OUT / 'graph_17_cross_cohort_degree.png'), width=700, height=500, scale=SCALE)
fig.show()

# Summary stats
print('\nCross-cohort degree per cohort:')
for c in ['BP', 'SZ', 'ASP']:
    vals = degree_df.loc[degree_df['cohort'] == c, 'cross_cohort_degree']
    print(f'  {c}: mean={vals.mean():.1f}, median={vals.median():.0f}, '
          f'max={vals.max():.0f}, 0-degree={int((vals == 0).sum())} ({(vals == 0).mean():.1%})')


Cross-cohort degree per cohort:
  BP: mean=107.8, median=104, max=241, 0-degree=0 (0.0%)
  SZ: mean=90.0, median=77, max=471, 0-degree=0 (0.0%)
  DR: mean=112.0, median=49, max=1069, 0-degree=0 (0.0%)
  ASP: mean=33.3, median=30, max=145, 0-degree=0 (0.0%)


### 13.9 Cohort-pair edge weight quality

For each cohort pair with cross-cohort edges, compare the **mean edge weight** of cross-cohort edges to the mean weight of intra-cohort edges within those same blocks. A ratio > 1.0 means cross-cohort edges are stronger than intra-cohort ones — a hallmark of genuine phenotypic overlap.

In [33]:
# Per cohort pair: mean cross-cohort weight vs mean intra-cohort weight per block
pair_quality = {}
for u, v, data in G.edges(data=True):
    bid = data.get('block', 'unknown')
    w = data.get('weight', 1.0)
    c1, c2 = node_cohort.get(u, '?').upper(), node_cohort.get(v, '?').upper()
    if c1 == c2:
        key = ('intra', bid, c1)
    else:
        pair = '-'.join(sorted([c1, c2]))
        key = ('cross', bid, pair)
    pair_quality.setdefault(key, []).append(w)

# Aggregate: per-block cross vs intra mean weights
block_weight_comparison = []
for bid in gs.edges_per_type:
    cross_w = []
    intra_w = []
    for key, weights in pair_quality.items():
        if key[1] != bid:
            continue
        if key[0] == 'cross':
            cross_w.extend(weights)
        else:
            intra_w.extend(weights)
    if cross_w and intra_w:
        ratio = np.mean(cross_w) / np.mean(intra_w)
        block_weight_comparison.append({
            'Block': bid,
            'Mean cross weight': round(np.mean(cross_w), 4),
            'Mean intra weight': round(np.mean(intra_w), 4),
            'Ratio (cross/intra)': round(ratio, 3),
            'Cross edges': len(cross_w),
            'Intra edges': len(intra_w),
        })

bwc_df = pd.DataFrame(block_weight_comparison).sort_values('Ratio (cross/intra)', ascending=False)

# Bar chart of weight ratios
fig = go.Figure(go.Bar(
    x=bwc_df['Ratio (cross/intra)'],
    y=bwc_df['Block'],
    orientation='h',
    marker_color=['#34d399' if r >= 1.0 else '#f87171' for r in bwc_df['Ratio (cross/intra)']],
    text=[f'{r:.2f}x' for r in bwc_df['Ratio (cross/intra)']],
    textposition='outside',
    textfont=dict(size=10),
))
fig.add_vline(x=1.0, line_dash='dash', line_color='white', opacity=0.4,
              annotation_text='Parity', annotation_position='top right')
fig.update_layout(
    title='Cross-cohort vs intra-cohort mean edge weight per block',
    xaxis_title='Mean cross-cohort weight / Mean intra-cohort weight',
    yaxis_title='',
    height=500, width=900,
    yaxis=dict(tickfont=dict(size=10)),
    **DARK,
)
fig.write_image(str(OUT / 'graph_18_weight_quality.png'), width=900, height=500, scale=SCALE)
fig.show()

print('\nCross vs intra weight comparison (blocks with cross-cohort edges):')
print(bwc_df.to_string(index=False))


Cross vs intra weight comparison (blocks with cross-cohort edges):
              Block  Mean cross weight  Mean intra weight  Ratio (cross/intra)  Cross edges  Intra edges
psychiatric_history             0.4775             0.2604                1.833         1674         1237
anxiety_impulsivity             0.3956             0.2627                1.506         1505         2003
    transdiagnostic             0.1443             0.1122                1.287         3971         4889
    sleep_circadian             0.3142             0.3013                1.043         1750         1897
    suicide_history             0.1577             0.1559                1.011         3353         3801
     family_history             0.1322             0.1322                1.000         4891         3590
      comorbidities             0.0983             0.0983                1.000         7088         4324
          substance             0.0943             0.0943                1.000         8307 

## 14. Graph Analysis — Conclusions

### Data science conclusions

The multiplex graph reveals a **three-tier taxonomy** of clinical blocks:

#### Tier 1 — Strongly transdiagnostic
**Cognition** (assortativity −0.75), **neuropsych** (−0.74), **substance** (−0.08), and **comorbidities** (+0.07) form the backbone of cross-cohort connectivity. Cognition and neuropsych are remarkable: their **negative assortativity** means patients are *more likely* to connect to someone from a different cohort than their own. This is not an artifact — BP and SZ patients scored on the same CVLT, TMT, and WAIS battery produce overlapping performance distributions, and kNN selects the genuinely nearest neighbours regardless of diagnosis.

#### Tier 2 — Selective bridges
**Mood**, **anxiety_impulsivity**, **biology**, **functioning**, **sleep_circadian**, **trauma**, and **psychiatric_history** produce fewer cross-cohort edges but they are **clinically informative** — connecting specific cohort pairs through shared instruments.

#### Tier 3 — Cohort-specific (assortativity = 1.0)
**Psychosis**, **insight**, **hostility_aggression**, **autism_profile** — these blocks use instruments administered to only one cohort. They serve a different function: **within-cohort resolution**, differentiating patients within each diagnostic group.

#### Key cohort-pair topology

| Cohort pair | Primary bridge domains | Relative strength |
|-------------|----------------------|-------------------|
| **BP ↔ SZ** | cognition, neuropsych, functioning | Dominant (largest bridge) |
| **ASP ↔ {BP, SZ}** | comorbidities, substance, demographics, functioning | Thin, generic |

#### Implications for embedding learning
1. **Weight normalization is essential** — without it, low-dimensional Gower blocks overwhelm sparse clinical blocks.
2. **First spectral components will separate ASP** from {BP, SZ} due to ASP's sparse cross-cohort connectivity.
3. **Intermediate components should reveal the BP↔SZ cognitive axis** — the negative-assortativity blocks contribute strong cross-cohort edges.
4. **0% patient isolation** — every patient has at least one edge, though ASP patients' positions will be determined primarily by thin universal blocks.

---

### Clinical conclusions

#### Cognition as the strongest transdiagnostic bridge
The strongest empirical link is **BP↔SZ through shared cognitive deficits** — consistent with the large body of literature showing cognitive impairment (executive function, processing speed, verbal memory) as a shared endophenotype across the schizophrenia–bipolar spectrum. A BP patient with severe executive dysfunction is more likely to connect to a similarly impaired SZ patient than to a cognitively spared BP patient. The graph encodes what the RDoC framework calls a *dimensional cognitive system deficit* that cuts across categorical diagnoses.

#### Treatment follows nosology
Treatment (assortativity +0.655) is the most assortative multi-cohort block — medication regimens are prescribed **based on diagnosis**, not transdiagnostic symptom profiles. If downstream clustering finds mixed BP+SZ clusters (through cognition), those clusters will have **heterogeneous treatment profiles**, raising the hypothesis-generating question of whether cognitive-profile-matched treatment strategies could improve outcomes.

#### ASP isolation reflects genuine measurement heterogeneity
ASP connects to other cohorts only through thin universal blocks. This reflects the fundamental reality that autism assessment uses a completely different clinical battery. The bridge would likely strengthen substantially if the consortium added a shared cognitive battery for ASP.

#### The dual structure
The multiplex encodes two complementary roles:
- **Bridging blocks** test the transdiagnostic hypothesis: do some BP patients resemble some SZ patients more than other BP patients?
- **Cohort-specific blocks** provide within-cohort resolution: within SZ, separating patients with good vs poor insight, predominantly positive vs negative symptoms.

Both are essential for clinically meaningful clustering. The graph is an honest representation of what the FACE data supports — not a model, but a **structured statement about observed similarity**.